# BAR Economy Solver Timing Comparison

This notebook compares the performance of different economy processing approaches:

1. **ProcessEconomy** (Option A): Engine calls Lua with team data, Lua returns modified data
2. **ResourceExcess** (Option B): Engine fires `ResourceExcess` callin, Lua handles excess directly

## Metrics Tracked

- **PreMunge**: Time to prepare data before solver
- **Solver**: Time in the waterfill algorithm
- **PostMunge**: Time to format results
- **PolicyCache**: Time to update transfer policy cache
- **CppSetters**: Time in C++ to apply Lua results (engine-side)


In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

DB_PATH = 'audit_logs.db'

# Colors for different metrics
METRIC_COLORS = {
    'PreMunge': '#4FC3F7',
    'Solver': '#EF5350',
    'PostMunge': '#66BB6A',
    'PolicyCache': '#FFD54F',
    'CppSetters': '#AB47BC',
    'BuildTeamData': '#4FC3F7',
    'WaterfillSolver': '#EF5350',
    'ApplyResults': '#66BB6A',
}

def get_conn():
    return sqlite3.connect(DB_PATH, timeout=10)

def query_df(sql, params=None):
    conn = get_conn()
    try:
        return pd.read_sql_query(sql, conn, params=params)
    finally:
        conn.close()

print("✓ Timing comparison notebook loaded")


✓ Timing comparison notebook loaded


## 1. Data Overview


In [2]:
# Check available timing data
timing_df = query_df("""
    SELECT metric, COUNT(*) as count, 
           AVG(time_us) as avg_us, 
           MIN(time_us) as min_us,
           MAX(time_us) as max_us,
           MIN(frame) as first_frame,
           MAX(frame) as last_frame
    FROM solver_audit
    GROUP BY metric
    ORDER BY avg_us DESC
""")

display(HTML("<h3>📊 Solver Timing Summary</h3>"))
if not timing_df.empty:
    display(timing_df.style.format({
        'avg_us': '{:.2f}',
        'min_us': '{:.2f}',
        'max_us': '{:.2f}'
    }))
else:
    print("No timing data found. Run the game with audit logging enabled.")


No timing data found. Run the game with audit logging enabled.


## 2. Timing Over Time

Individual charts for each metric to identify anomalies:


In [3]:
def plot_timing_by_metric(limit=5000):
    """Plot each timing metric in its own chart for clarity."""
    
    solver_df = query_df(f"""
        SELECT frame, metric, time_us FROM solver_audit
        WHERE frame > (SELECT MAX(frame) FROM solver_audit) - {limit}
        AND metric != 'Overall'
        ORDER BY frame
    """)
    
    if solver_df.empty:
        print("No solver timing data available")
        return
    
    metrics = solver_df['metric'].unique()
    n_metrics = len(metrics)
    
    fig, axes = plt.subplots(n_metrics, 1, figsize=(14, 3 * n_metrics), sharex=True)
    fig.patch.set_facecolor('#16213e')
    
    if n_metrics == 1:
        axes = [axes]
    
    for ax, metric in zip(axes, metrics):
        ax.set_facecolor('#1a1a2e')
        ax.tick_params(colors='white')
        
        mdf = solver_df[solver_df['metric'] == metric]
        color = METRIC_COLORS.get(metric, '#888888')
        
        ax.plot(mdf['frame'], mdf['time_us'], color=color, alpha=0.7, linewidth=0.8)
        
        # Add rolling average
        if len(mdf) > 30:
            rolling_avg = mdf['time_us'].rolling(window=30).mean()
            ax.plot(mdf['frame'], rolling_avg, color='white', alpha=0.9, linewidth=1.5, label='30-frame avg')
        
        # Stats
        avg_time = mdf['time_us'].mean()
        max_time = mdf['time_us'].max()
        p95 = mdf['time_us'].quantile(0.95)
        
        ax.axhline(y=avg_time, color='#66BB6A', linestyle='--', alpha=0.5, label=f'Avg: {avg_time:.1f}μs')
        ax.axhline(y=p95, color='#FFD54F', linestyle=':', alpha=0.5, label=f'P95: {p95:.1f}μs')
        
        ax.set_title(f'{metric}', color=color, fontsize=12)
        ax.set_ylabel('Time (μs)', color='white')
        ax.legend(loc='upper right', facecolor='#1a1a2e', edgecolor='#444', labelcolor='white', fontsize=8)
        
        # Flag suspicious values (powers of 2 might indicate timer issues)
        suspicious = mdf[mdf['time_us'] > 1000]
        if not suspicious.empty:
            ax.scatter(suspicious['frame'], suspicious['time_us'], color='#EF5350', s=20, zorder=5)
    
    axes[-1].set_xlabel('Frame', color='white')
    fig.suptitle(f'⏱️ Solver Timing by Metric (Last {limit} Frames)', color='white', fontsize=14)
    plt.tight_layout()
    plt.show()

plot_timing_by_metric()


No solver timing data available


## 3. Timing Distribution Histograms

Understand the distribution of timing values and identify outliers:


In [4]:
def plot_timing_histograms():
    """Histogram of timing values per metric."""
    
    solver_df = query_df("""
        SELECT metric, time_us FROM solver_audit
        WHERE metric != 'Overall'
    """)
    
    if solver_df.empty:
        print("No solver timing data available")
        return
    
    metrics = solver_df['metric'].unique()
    n_metrics = len(metrics)
    cols = min(3, n_metrics)
    rows = (n_metrics + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    fig.patch.set_facecolor('#16213e')
    axes = np.array(axes).flatten() if n_metrics > 1 else [axes]
    
    for ax, metric in zip(axes, metrics):
        ax.set_facecolor('#1a1a2e')
        ax.tick_params(colors='white')
        
        mdf = solver_df[solver_df['metric'] == metric]['time_us']
        color = METRIC_COLORS.get(metric, '#888888')
        
        # Use log bins for wide-ranging data
        if mdf.max() > mdf.min() * 10 and mdf.min() > 0:
            bins = np.logspace(np.log10(max(0.1, mdf.min())), np.log10(mdf.max()), 50)
            ax.set_xscale('log')
        else:
            bins = 50
        
        ax.hist(mdf, bins=bins, color=color, alpha=0.7, edgecolor='white', linewidth=0.3)
        
        # Stats lines
        ax.axvline(mdf.mean(), color='#66BB6A', linestyle='--', label=f'Mean: {mdf.mean():.1f}')
        ax.axvline(mdf.median(), color='#FFD54F', linestyle=':', label=f'Median: {mdf.median():.1f}')
        
        ax.set_title(metric, color=color)
        ax.set_xlabel('Time (μs)', color='white')
        ax.set_ylabel('Count', color='white')
        ax.legend(fontsize=8, facecolor='#1a1a2e', labelcolor='white')
    
    # Hide unused axes
    for ax in axes[len(metrics):]:
        ax.set_visible(False)
    
    fig.suptitle('📊 Timing Distributions', color='white', fontsize=14)
    plt.tight_layout()
    plt.show()

plot_timing_histograms()


No solver timing data available


## 4. Anomaly Detection

Find frames with unusually high timing (potential performance issues):


In [5]:
def find_timing_anomalies(threshold_percentile=99):
    """Find frames with timing above the given percentile."""
    
    solver_df = query_df("""
        SELECT frame, metric, time_us FROM solver_audit
        WHERE metric != 'Overall'
    """)
    
    if solver_df.empty:
        print("No timing data available")
        return
    
    display(HTML(f"<h3>🚨 Timing Anomalies (>{threshold_percentile}th percentile)</h3>"))
    
    anomalies = []
    for metric in solver_df['metric'].unique():
        mdf = solver_df[solver_df['metric'] == metric]
        threshold = mdf['time_us'].quantile(threshold_percentile / 100)
        
        high_frames = mdf[mdf['time_us'] > threshold]
        for _, row in high_frames.iterrows():
            anomalies.append({
                'Frame': row['frame'],
                'Metric': metric,
                'Time (μs)': row['time_us'],
                'Threshold': threshold
            })
    
    if anomalies:
        anomaly_df = pd.DataFrame(anomalies).sort_values('Time (μs)', ascending=False).head(20)
        display(anomaly_df.style.format({'Time (μs)': '{:.2f}', 'Threshold': '{:.2f}'}))
        
        # Check for suspicious power-of-2 values
        pow2_values = [2**i for i in range(8, 14)]  # 256, 512, 1024, 2048, 4096, 8192
        suspicious = anomaly_df[anomaly_df['Time (μs)'].apply(lambda x: any(abs(x - p) < 10 for p in pow2_values))]
        if not suspicious.empty:
            display(HTML("<p style='color:#EF5350;'>⚠️ Some values are close to powers of 2, which may indicate timer resolution issues.</p>"))
    else:
        print("No anomalies found above threshold.")

find_timing_anomalies(99)


No timing data available
